 ==========================================================

 InsightForge AI

 Agentic AI for Intelligent Information Extraction

 Author : Sumit Yagik

 Version : 1.0.0

 ==========================================================

# InsightForge AI

## Objective

Build a production-inspired Agentic AI system capable of:

- Keyword Extraction
- Named Entity Recognition (NER)
- Intelligent Tool Selection
- Structured JSON Output
- Future RAG Integration

---

## Tech Stack

- Python
- Google Colab
- LangChain
- Groq
- KeyBERT
- spaCy
- Sentence Transformers

In [1]:
!pip install -qU \
langchain \
langchain-core \
langchain-community \
langchain-groq \
langgraph \
groq \
transformers==4.41.2 \
sentence-transformers==3.0.1 \
keybert==0.8.5 \
spacy \
faiss-cpu \
python-dotenv \
pydantic \
peft==0.11.1 \
tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 568.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 

In [2]:
# ==========================================================
# STANDARD LIBRARIES
# ==========================================================

import os
import warnings
from typing import Dict, List


# ==========================================================
# LANGCHAIN
# ==========================================================

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

# ==========================================================
# NLP
# ==========================================================

from sentence_transformers import SentenceTransformer
from keybert import KeyBERT
import spacy

# ==========================================================
# LLM
# ==========================================================

from langchain_groq import ChatGroq

# ==========================================================
# GOOGLE COLAB
# ==========================================================

from google.colab import userdata

In [3]:
# ==========================================================
# PROJECT CONFIGURATION
# ==========================================================

warnings.filterwarnings("ignore")

PROJECT_NAME = "InsightForge AI"
PROJECT_VERSION = "1.0.0"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
SPACY_MODEL_NAME = "en_core_web_sm"
LLM_MODEL_NAME = "llama-3.3-70b-versatile"

In [4]:
#=============================================================================
# APPLICATION CONFIGURATION
#=============================================================================

class AppConfig:
  PROJECT_NAME = "InsightForge AI"
  PROJECT_VERSION = "1.0.0"
  EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
  SPACY_MODEL = "en_core_web_sm"
  LLM_MODEL = "llama-3.3-70b-versatile"
  MAX_KEYWORDS = 10

config = AppConfig()

In [8]:
# ==========================================================
# MODEL REGISTRY
# ==========================================================

class ModelRegistry:
    """
    Central registry for all AI models.
    Models are loaded lazily (only when needed).
    """

    def __init__(self):
        self.embedding_model = None
        self.keyword_model = None
        self.nlp = None
        self.llm = None
        self.summarizer = None

    #embedding model function
    def get_embedding_model(self):
      """
      Load the SentenceTransformer model only once.
      """

      if self.embedding_model is None:
        print("Loading Embedding Model...")

        self.embedding_model=SentenceTransformer(
            config.EMBEDDING_MODEL
        )
        print("Embedding Model Loaded Successfully!")
      else:
        print("Using Cached Embedding Model.")

      return self.embedding_model

    #keyword model function
    def get_keyword_model(self):
      """
      Load the KeyBERT model only once.
      """

      if self.keyword_model is None:
        print("Loading KeyBERT Model...")
        embedding_model = self.get_embedding_model()

        self.keyword_model = KeyBERT(
            model = embedding_model
            )
        print("KeyBERT Model Loaded Successfully!")

      else:
        print("Using Cached KeyBERT Model.")

      return self.keyword_model

    #NER tool function
    def get_spacy_model(self):
      """
      Load the spaCy model only once.
      """

      if self.nlp is None:

          print("Loading spaCy Model...")

          self.nlp = spacy.load(config.SPACY_MODEL)

          print("spaCy Model Loaded Successfully!")

      else:

          print("Using Cached spaCy Model.")

      return self.nlp


    #Summarization tool function
    def get_summarizer(self):

      if self.summarizer is None:

          print("Loading Summarizer...")

          self.summarizer = pipeline(
              "summarization",
              model="facebook/bart-large-cnn"
          )

      return self.summarizer

registry = ModelRegistry()


In [6]:
embedding_model = registry.get_embedding_model()

Loading Embedding Model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded Successfully!


In [9]:
keyword_model = registry.get_keyword_model()

Loading KeyBERT Model...
Loading Embedding Model...
Embedding Model Loaded Successfully!
KeyBERT Model Loaded Successfully!


In [10]:
nlp = registry.get_spacy_model()

Loading spaCy Model...
spaCy Model Loaded Successfully!


In [11]:
print(dir(registry))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'embedding_model', 'get_embedding_model', 'get_keyword_model', 'get_spacy_model', 'get_summarizer', 'keyword_model', 'llm', 'nlp', 'summarizer']


In [12]:
import time

In [13]:
from langchain_core.tools import tool

In [14]:
#Keyword_tool

@tool
def keyword_tool(text: str) -> dict:
    """
    Extract important keywords from text using KeyBERT.
    """
    if not isinstance(text, str):
        return {
            "status": "error",
            "tool_name": "keyword_tool",
            "execution_time": 0,
            "data": None,
            "metadata": None,
            "error": "Input must be a string."
        }

    if not text.strip():
        return {
            "status": "error",
            "tool_name": "keyword_tool",
            "execution_time": 0,
            "data": None,
            "metadata": None,
            "error": "Input text is empty."
        }

    # -----------------------------
    # Start Timer
    # -----------------------------
    start_time = time.time()

    # -----------------------------
    # Get KeyBERT Model
    # -----------------------------
    keyword_model = registry.get_keyword_model()

    # -----------------------------
    # Extract Keywords
    # -----------------------------
    keywords = keyword_model.extract_keywords(
        text,
        top_n=config.MAX_KEYWORDS,
        keyphrase_ngram_range=(1, 2),
        stop_words="english"
    )

    # -----------------------------
    # Convert Output
    # -----------------------------
    formatted_keywords = []

    for keyword, score in keywords:
        formatted_keywords.append(
            {
                "keyword": keyword,
                "score": round(float(score), 3)
            }
        )

    # -----------------------------
    # Stop Timer
    # -----------------------------
    #execution_time = round(time.time() - start_time, 3)

    # -----------------------------
    # Return Structured Response
    # -----------------------------
    return {
        "status": "success",
        "tool_name": "keyword_tool",
        "execution_time": round(time.time() - start_time, 3),

        "data": {
            "keywords": formatted_keywords
        },

        "metadata": {
            "keyword_count": len(formatted_keywords),
            "model": "KeyBERT"
        },

        "error": None
    }

In [15]:
keyword_tool.invoke({"text": "Deep learning is transforming image analysis."})

Using Cached KeyBERT Model.


{'status': 'success',
 'tool_name': 'keyword_tool',
 'execution_time': 7.265,
 'data': {'keywords': [{'keyword': 'learning transforming', 'score': 0.579},
   {'keyword': 'transforming image', 'score': 0.546},
   {'keyword': 'deep learning', 'score': 0.541},
   {'keyword': 'transforming', 'score': 0.51},
   {'keyword': 'image analysis', 'score': 0.446},
   {'keyword': 'learning', 'score': 0.371},
   {'keyword': 'image', 'score': 0.343},
   {'keyword': 'deep', 'score': 0.314},
   {'keyword': 'analysis', 'score': 0.271}]},
 'metadata': {'keyword_count': 9, 'model': 'KeyBERT'},
 'error': None}

In [16]:
keyword_tool.invoke({"text": ""})

{'status': 'error',
 'tool_name': 'keyword_tool',
 'execution_time': 0,
 'data': None,
 'metadata': None,
 'error': 'Input text is empty.'}

In [17]:
#Testing
sample_text = """
Deep learning has revolutionized computer vision.
Convolutional Neural Networks are widely used for image classification.
"""
result = keyword_tool.invoke(
    {
        "text": sample_text
    }
)

print(result)

Using Cached KeyBERT Model.
{'status': 'success', 'tool_name': 'keyword_tool', 'execution_time': 0.131, 'data': {'keywords': [{'keyword': 'deep learning', 'score': 0.577}, {'keyword': 'convolutional neural', 'score': 0.576}, {'keyword': 'vision convolutional', 'score': 0.565}, {'keyword': 'image classification', 'score': 0.496}, {'keyword': 'computer vision', 'score': 0.481}, {'keyword': 'convolutional', 'score': 0.467}, {'keyword': 'learning revolutionized', 'score': 0.453}, {'keyword': 'neural networks', 'score': 0.449}, {'keyword': 'classification', 'score': 0.386}, {'keyword': 'neural', 'score': 0.385}]}, 'metadata': {'keyword_count': 10, 'model': 'KeyBERT'}, 'error': None}


In [18]:
import time
from langchain_core.tools import tool

@tool
def ner_tool(text: str) -> dict:
    """
    Extract named entities from text using spaCy.
    """

    # Validation
    if not isinstance(text, str):
        return {
            "status": "error",
            "tool_name": "ner_tool",
            "execution_time": 0,
            "data": None,
            "metadata": None,
            "error": "Input must be a string."
        }

    if not text.strip():
        return {
            "status": "error",
            "tool_name": "ner_tool",
            "execution_time": 0,
            "data": None,
            "metadata": None,
            "error": "Input text is empty."
        }

    start_time = time.time()

    nlp = registry.get_spacy_model()

    doc = nlp(text)

    entities = []

    for ent in doc.ents:
        entities.append({
            "text": ent.text,
            "label": ent.label_
        })

    #execution_time = round(time.time() - start_time, 3)

    return {
        "status": "success",
        "tool_name": "ner_tool",
        "execution_time": round(time.time() - start_time, 3),
        "data": {
            "entities": entities
        },
        "metadata": {
            "entity_count": len(entities),
            "model": "spaCy"
        },
        "error": None
    }


In [19]:
# Testing
sample_text = """
Google CEO Sundar Pichai visited India and met Prime Minister Narendra Modi in New Delhi.
"""
result = ner_tool.invoke({
    "text": sample_text
})

print(result)

Using Cached spaCy Model.
{'status': 'success', 'tool_name': 'ner_tool', 'execution_time': 0.03, 'data': {'entities': [{'text': 'Google', 'label': 'ORG'}, {'text': 'Sundar Pichai', 'label': 'PERSON'}, {'text': 'India', 'label': 'GPE'}, {'text': 'Narendra Modi', 'label': 'PERSON'}, {'text': 'New Delhi', 'label': 'GPE'}]}, 'metadata': {'entity_count': 5, 'model': 'spaCy'}, 'error': None}


In [20]:
from transformers import pipeline

In [21]:
@tool
def summarize_tool(text: str) -> dict:
    """
    Generate a concise summary of the given text using the BART summarization model.
    This tool is useful for summarizing research papers, articles, reports,
    and other long documents while preserving the main ideas.
    """

    summarizer = registry.get_summarizer()

    input_words = len(text.split())

    max_len = min(120, max(30, input_words // 2))
    min_len = min(20, max_len - 5)

    summary = summarizer(
        text,
        max_length=max_len,
        min_length=min_len,
        do_sample=False
    )[0]["summary_text"]

    start_time = time.time()

    return {
      "status": "success",
      "tool_name": "summarize_tool",
      "execution_time": round(time.time() - start_time, 3),
      "data": {
          "summary": summary
      },
      "metadata": {
          "model": "facebook/bart-large-cnn"
      },
      "error": None
  }

In [22]:
# ==========================================================
# LOAD LLM
# ==========================================================

llm = ChatGroq(
    api_key=userdata.get("GROQ_API_KEY2"),
    model=config.LLM_MODEL,
    temperature=0
)

print("Groq LLM Loaded Successfully!")

Groq LLM Loaded Successfully!


In [23]:
tools = [
    keyword_tool,
    ner_tool,
    summarize_tool
]

print(tools)

[StructuredTool(name='keyword_tool', description='Extract important keywords from text using KeyBERT.', args_schema=<class 'langchain_core.utils.pydantic.keyword_tool'>, func=<function keyword_tool at 0x7caec0a09b20>), StructuredTool(name='ner_tool', description='Extract named entities from text using spaCy.', args_schema=<class 'langchain_core.utils.pydantic.ner_tool'>, func=<function ner_tool at 0x7caebcd7b600>), StructuredTool(name='summarize_tool', description='Generate a concise summary of the given text using the BART summarization model.\nThis tool is useful for summarizing research papers, articles, reports,\nand other long documents while preserving the main ideas.', args_schema=<class 'langchain_core.utils.pydantic.summarize_tool'>, func=<function summarize_tool at 0x7caebcd7b240>)]


In [24]:
import langchain
print(langchain.__version__)

1.3.14


In [25]:
!pip -q install langgraph

In [26]:
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

In [27]:
agent = create_react_agent(
    model=llm,
    tools=tools
)

In [29]:
response = agent.invoke(
    {
        "messages":[
            (
                "user",
                """
Extract keywords from this text.

Deep learning is widely used in computer vision and image segmentation.
"""
            )
        ]
    }
)

Using Cached KeyBERT Model.


In [30]:
print(response)

{'messages': [HumanMessage(content='\nExtract keywords from this text.\n\nDeep learning is widely used in computer vision and image segmentation.\n', additional_kwargs={}, response_metadata={}, id='3a0749b6-e73a-420e-aa19-9a7195c01db7'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '4mnf1ggjm', 'function': {'arguments': '{"text":"Deep learning is widely used in computer vision and image segmentation."}', 'name': 'keyword_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 412, 'total_tokens': 437, 'completion_time': 0.06498147, 'completion_tokens_details': None, 'prompt_time': 0.037305945, 'prompt_tokens_details': None, 'queue_time': 0.056754114, 'total_time': 0.102287415}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7461-03a9-7400-8377-fa053a63900a-0', tool_

In [31]:
response = agent.invoke(
    {
        "messages":[
            (
                "user", """Extract all named entities. Google CEO Sundar Pichai visited India."""
            )
        ]
    }
)

print(response)

Using Cached spaCy Model.
{'messages': [HumanMessage(content='Extract all named entities. Google CEO Sundar Pichai visited India.', additional_kwargs={}, response_metadata={}, id='d627fc76-46e9-4184-925c-83491752f22c'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '941an863d', 'function': {'arguments': '{"text":"Google CEO Sundar Pichai visited India."}', 'name': 'ner_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 409, 'total_tokens': 432, 'completion_time': 0.069145166, 'completion_tokens_details': None, 'prompt_time': 0.04652915, 'prompt_tokens_details': None, 'queue_time': 0.056841689, 'total_time': 0.115674316}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7461-0d8f-7aa3-9cc8-94bc6c0f7a68-0', tool_calls=[{'name': 'ner_tool', 'args': {'text': 'Google C

In [32]:
# Testing agent : 1
paper = """
Google DeepMind introduced Gemini, a multimodal large language model.
The research focuses on reasoning, planning and vision-language understanding.
The work was led by Demis Hassabis in London.
"""

response = agent.invoke(
    {
        "messages":[
            (
                "user",
                f"""
Analyze the following research text. Extract important keywords. Also identify all named entities.
Text: {paper}
"""
            )
        ]
    }
)

print(response)

Using Cached KeyBERT Model.
Using Cached spaCy Model.
{'messages': [HumanMessage(content='\nAnalyze the following research text. Extract important keywords. Also identify all named entities.\nText: \nGoogle DeepMind introduced Gemini, a multimodal large language model.\nThe research focuses on reasoning, planning and vision-language understanding.\nThe work was led by Demis Hassabis in London.\n\n', additional_kwargs={}, response_metadata={}, id='c815cdb3-c65a-48d3-af0d-468ac01d5d2e'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hn12r96hz', 'function': {'arguments': '{"text":"Google DeepMind introduced Gemini, a multimodal large language model. The research focuses on reasoning, planning and vision-language understanding. The work was led by Demis Hassabis in London."}', 'name': 'keyword_tool'}, 'type': 'function'}, {'id': 'f0phtg13n', 'function': {'arguments': '{"text":"Google DeepMind introduced Gemini, a multimodal large language model. The research focuses on re

In [33]:
# TEsting agent : 2
paper = """
Large Language Models have become increasingly important in medical diagnosis.
Researchers from Stanford University and Microsoft developed a framework for clinical reasoning.
"""

response = agent.invoke(
    {
        "messages":[
            (
                "user",
                f"Analyze this research abstract:\n\n{paper}"
            )
        ]
    }
)

print(response)

Using Cached KeyBERT Model.
Using Cached spaCy Model.
Loading Summarizer...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Your max_length is set to 30, but your input_length is only 26. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=13)


{'messages': [HumanMessage(content='Analyze this research abstract:\n\n\nLarge Language Models have become increasingly important in medical diagnosis.\nResearchers from Stanford University and Microsoft developed a framework for clinical reasoning.\n', additional_kwargs={}, response_metadata={}, id='c1dfa9fd-47b7-42fa-91f4-369b96a53aa9'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '27s7bx9gg', 'function': {'arguments': '{"text":"Large Language Models have become increasingly important in medical diagnosis. Researchers from Stanford University and Microsoft developed a framework for clinical reasoning."}', 'name': 'keyword_tool'}, 'type': 'function'}, {'id': 'w4m3nyd9x', 'function': {'arguments': '{"text":"Large Language Models have become increasingly important in medical diagnosis. Researchers from Stanford University and Microsoft developed a framework for clinical reasoning."}', 'name': 'ner_tool'}, 'type': 'function'}, {'id': '1fa2bdy8t', 'function': {'argument

In [34]:
# Testing agent : 3
sample_text = """
Google DeepMind recently introduced Gemini, a multimodal AI model capable of
reasoning across text, images, audio, and code.

The project was led by Demis Hassabis in London.

Researchers explained how deep learning and transformer architectures improve
large-scale reasoning and image understanding.

The research also discusses reinforcement learning and computer vision.
"""

response = agent.invoke(
    {
        "messages":[
            (
                "user",
                f"""
Summarize this text.

{sample_text}
"""
            )
        ]
    }
)

print(response["messages"][-1].content)

Google DeepMind recently introduced Gemini, a multimodal AI model. Gemini is capable of reasoning across text, images, audio, and code. The project was led by Demis Hassabis in London. Researchers explained how deep learning and transformer architectures improve large-scale reasoning and image understanding. The research also discusses reinforcement learning and computer vision.


In [35]:
sample_text = """
Google DeepMind recently introduced Gemini, a multimodal AI model capable of
reasoning across text, images, audio, and code.

The project was led by Demis Hassabis in London.

Researchers explained how deep learning and transformer architectures improve
large-scale reasoning and image understanding.

The research also discusses reinforcement learning and computer vision.
"""

result = summarize_tool.invoke(
    {
        "text": sample_text
    }
)

print(result)

{'status': 'success', 'tool_name': 'summarize_tool', 'execution_time': 0.0, 'data': {'summary': 'Google DeepMind recently introduced Gemini, a multimodal AI model capable ofreasoning across text, images, audio, and code.'}, 'metadata': {'model': 'facebook/bart-large-cnn'}, 'error': None}


# A small chatbot  
where user can ask his query and agent will summarize, extract keywords or extract entity.

In [36]:
print("="*50)
print("InsightForge AI")
print("Type 'exit' to quit")
print("="*50)

while True:

    user_input = input("\nYou : ")

    if user_input.lower() == "exit":
        break

    response = agent.invoke(
        {
            "messages":[
                ("user", user_input)
            ]
        }
    )

    print("\nAI :")
    print(response["messages"][-1].content)

InsightForge AI
Type 'exit' to quit

You : Hii! Whatsapp
Using Cached spaCy Model.
Using Cached KeyBERT Model.

AI :
The text "Hii! Whatsapp" does not contain any named entities. The important keywords extracted from the text are "hii whatsapp", "whatsapp", and "hii".

You : OpenAI recently introduced a new generation of language models capable of understanding and generating human-like text across multiple domains. These models are trained using transformer architectures and reinforcement learning techniques to improve reasoning, coding, and conversational abilities. Researchers including Sam Altman and Ilya Sutskever have contributed significantly to the advancement of generative AI. The technology is being adopted by organizations worldwide for education, healthcare, finance, and software development. However, experts also emphasize the importance of AI safety, responsible deployment, and alignment with human values. Extract KEywords.
Using Cached KeyBERT Model.
Using Cached spaCy M

#Using Transformers BART and Pipeline

In [37]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

text = """
Deep learning has revolutionized computer vision by enabling neural networks
to automatically learn features from large datasets.
"""

input_words = len(text.split())

max_len = min(120, max(30, input_words // 2))
min_len = min(20, max_len - 5)

summary = summarizer(
    text,
    max_length=max_len,
    min_length=min_len,
    do_sample=False
)

print(result)

Your max_length is set to 30, but your input_length is only 25. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=12)


{'status': 'success', 'tool_name': 'summarize_tool', 'execution_time': 0.0, 'data': {'summary': 'Google DeepMind recently introduced Gemini, a multimodal AI model capable ofreasoning across text, images, audio, and code.'}, 'metadata': {'model': 'facebook/bart-large-cnn'}, 'error': None}


In [38]:
def format_analysis(summary=None, keywords=None, entities=None):
    """
    Format the outputs from different AI tools into a readable report.
    """

    report = "\n"
    report += "=" * 60 + "\n"
    report += "🧠 InsightForge AI Analysis Report\n"
    report += "=" * 60 + "\n\n"

    if summary:
        report += "📄 SUMMARY\n"
        report += "-" * 60 + "\n"
        report += summary + "\n\n"

    if keywords:
        report += "🔑 KEYWORDS\n"
        report += "-" * 60 + "\n"

        for keyword in keywords:
          report += (
            f"• {keyword['keyword'].title()} "
            f"(Score: {keyword['score']:.3f})\n"
        )

        report += "\n"

    if entities:
        report += "👤 NAMED ENTITIES\n"
        report += "-" * 60 + "\n"

        for entity in entities:

          LABEL_MAP = {
            "ORG": "Organization",
            "PERSON": "Person",
            "GPE": "Location",
            "LOC": "Location",
            "DATE": "Date",
            "TIME": "Time",
        }
          label = LABEL_MAP.get(
              entity["label"],
              entity["label"]
          )

          report += f"• {entity['text']} ({label})\n"

        report += "\n"

    return report

In [39]:
# ==========================================================
# INSIGHTFORGE AI ORCHESTRATOR
# ==========================================================
def analyze_document(text: str):

    summary = None
    keywords = None
    entities = None

    # -------- Summary --------
    summary_result = summarize_tool.invoke({"text": text})

    if summary_result["status"] == "success":
        summary = summary_result["data"]["summary"]

    # -------- Keywords --------
    keyword_result = keyword_tool.invoke({"text": text})

    if keyword_result["status"] == "success":
        keywords = keyword_result["data"]["keywords"]

    # -------- Entities --------
    ner_result = ner_tool.invoke({"text": text})

    if ner_result["status"] == "success":
        entities = ner_result["data"]["entities"]

    report = format_analysis(
        summary,
        keywords,
        entities
    )

    return report

In [40]:
#Testing : 4
sample_text = """
Google DeepMind recently introduced Gemini, a multimodal AI model capable of reasoning across text, images, audio, and code.

The project was led by Demis Hassabis in London.

Researchers explained how deep learning and transformer architectures improve large-scale reasoning and image understanding.

The research also discusses reinforcement learning and computer vision.
"""
report = analyze_document(sample_text)

print(report)

Using Cached KeyBERT Model.
Using Cached spaCy Model.

🧠 InsightForge AI Analysis Report

📄 SUMMARY
------------------------------------------------------------
Google DeepMind recently introduced Gemini, a multimodal AI model capable of reasoning across text, images, audio, and code. The

🔑 KEYWORDS
------------------------------------------------------------
• Google Deepmind (Score: 0.735)
• Deepmind (Score: 0.674)
• Deepmind Recently (Score: 0.643)
• Multimodal Ai (Score: 0.611)
• Deep Learning (Score: 0.520)
• Gemini Multimodal (Score: 0.506)
• Ai (Score: 0.505)
• Reasoning Image (Score: 0.497)
• Ai Model (Score: 0.492)
• Multimodal (Score: 0.489)

👤 NAMED ENTITIES
------------------------------------------------------------
• Google DeepMind (Organization)
• Gemini (Location)
• Demis Hassabis (Organization)
• London (Location)




Google DeepMind recently introduced Gemini 2.5 Pro, a multimodal artificial intelligence model capable of understanding text, images, audio, video, and source code. The research was led by Demis Hassabis and Jeff Dean in collaboration with teams based in London and California. The model combines transformer architectures, reinforcement learning, retrieval-augmented generation, and large-scale distributed training to improve reasoning, scientific discovery, and software engineering tasks. Researchers evaluated the model on benchmarks involving mathematics, programming, biomedical research, and multilingual understanding. The technology is expected to accelerate innovation in healthcare, robotics, climate science, and education while maintaining a strong focus on AI safety, responsible deployment, and transparency.

In [41]:
#Testing : 5
sample_text = """
Climate change continues to affect ecosystems around the world through rising temperatures, melting glaciers, and increasing sea levels.
Scientists from the Intergovernmental Panel on Climate Change (IPCC) have warned that global greenhouse gas emissions must be reduced significantly over the next decade.
Renewable energy technologies such as solar panels, wind farms, and electric vehicles are expected to play a crucial role in reducing carbon emissions and achieving sustainable development goals.
"""
report = analyze_document(sample_text)

print(report)

Using Cached KeyBERT Model.
Using Cached spaCy Model.

🧠 InsightForge AI Analysis Report

📄 SUMMARY
------------------------------------------------------------
Scientists from the Intergovernmental Panel on Climate Change (IPCC) have warned that global greenhouse gas emissions must be reduced significantly over the next decade.Renewable

🔑 KEYWORDS
------------------------------------------------------------
• Climate Change (Score: 0.630)
• Carbon Emissions (Score: 0.549)
• Emissions Reduced (Score: 0.546)
• Global Greenhouse (Score: 0.544)
• Emissions Achieving (Score: 0.533)
• Reducing Carbon (Score: 0.507)
• Achieving Sustainable (Score: 0.491)
• Greenhouse Gas (Score: 0.485)
• Renewable Energy (Score: 0.455)
• Emissions (Score: 0.452)

👤 NAMED ENTITIES
------------------------------------------------------------
• the Intergovernmental Panel on Climate Change (Organization)
• the next decade (Date)




In [42]:
#Testing : 6
sample_text = """
Google DeepMind recently introduced Gemini 2.5 Pro, a multimodal artificial intelligence model capable of understanding text, images, audio, video, and source code.
The research was led by Demis Hassabis and Jeff Dean in collaboration with teams based in London and California.
The model combines transformer architectures, reinforcement learning, retrieval-augmented generation, and large-scale distributed training to improve reasoning, scientific discovery, and software engineering tasks.
Researchers evaluated the model on benchmarks involving mathematics, programming, biomedical research, and multilingual understanding.
The technology is expected to accelerate innovation in healthcare, robotics, climate science, and education while maintaining a strong focus on AI safety, responsible deployment, and transparency.
"""
report = analyze_document(sample_text)

print(report)

Using Cached KeyBERT Model.
Using Cached spaCy Model.

🧠 InsightForge AI Analysis Report

📄 SUMMARY
------------------------------------------------------------
Gemini 2.5 Pro is a multimodal artificial intelligence model capable of understanding text, images, audio, video, and source code. The technology is expected to accelerate innovation in healthcare, robotics, climate science, and education.

🔑 KEYWORDS
------------------------------------------------------------
• Google Deepmind (Score: 0.724)
• Deepmind Recently (Score: 0.645)
• Deepmind (Score: 0.643)
• Ai (Score: 0.467)
• Artificial Intelligence (Score: 0.450)
• Ai Safety (Score: 0.403)
• Multimodal (Score: 0.399)
• Focus Ai (Score: 0.393)
• Pro Multimodal (Score: 0.392)
• Intelligence Model (Score: 0.376)

👤 NAMED ENTITIES
------------------------------------------------------------
• Google DeepMind (Organization)
• Gemini 2.5 Pro (WORK_OF_ART)
• Demis Hassabis (Organization)
• Jeff Dean (Person)
• London (Location)
• Cali